# 09 pamoka – Metakognicijos dizaino šablonas


## Setup

This notebook demonstrates the Metacognition design pattern using the Microsoft Agent Framework.

**Prerequisites:**
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kas yra metakognicija?

Metakognicija yra **mąstymas apie mąstymą**. Dirbtinio intelekto agentų kontekste tai reiškia kurti agentus, kurie gali:

- **Savarankiškai reflektuoti** savo rezultatus ir samprotavimo procesą
- **Aptikti klaidas** ir jas tinkamai tvarkyti, o ne tyliai nukrypti nuo tikslo
- **Vertinti**, ar jų atsakymai yra pilni ir naudingi
- **Prisitaikyti** prie strategijos, kai pradinė metodika neveikia (pvz., pereiti prie atsarginės sistemos)

Metakognityvus agentas ne tik atsako į klausimus — jis stebi savo veiklą ir koreguojasi realiu laiku.


## Pagrindiniai ir atsarginiai įrankiai

Dažnas metakognityvinis modelis yra **atsarginio plano strategija**. Agentas pirmiausia bando naudoti pagrindinį įrankį; jei jis nepavyksta (pvz., 404 klaida), agentas atpažįsta klaidą ir skaidriai pereina prie atsarginio įrankio.

Tai atspindi realius sistemas, kuriose pagrindinės paslaugos gali būti neprieinamos, o agentai turi patys diagnozuoti problemą prieš pasirinkdami alternatyvų kelią.

Toliau apibrėžiame du skrydžių paieškos įrankius:
- **Pagrindinis** — apima Paryžių, Tokiją ir Barseloną
- **Atsarginis** — apima Berlyną, Sidnėjų ir Niujorką


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Savianalizuojantis agentas su klaidų atkūrimu

Žemiau pateiktas agentas yra nurodytas pirmiausia išbandyti pagrindinę skrydžio sistemą, atpažinti gedimus ir skaidriai pereiti prie atsarginės sistemos. Po kiekvieno atsakymo jis trumpai save įvertina, ar visiškai atsakė į vartotojo klausimą.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Savarankiško Vertinimo Šablonas

Kitas metakognicijos aspektas yra **savarankiškas vertinimas**: atskiras agentas (ar tas pats agentas antrą kartą peržiūrint) įvertina atsakymą pagal išsamumą, tikslumą ir naudingumą.

Žemiau sukuriame `ResponseEvaluator` agentą, kuris vertina kelionių agento atsakymus trimis aspektais.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Santrauka

Šioje pamokoje išmokote, kaip kurti **metakognityvinius agentus** naudojant Microsoft Agent Framework:

- **Savi-refleksija**: Agentai, kurie stebi savo sprendimų priėmimą ir skaidriai komunikuoja, kas įvyko.
- **Klaidų atkūrimas naudojant atsarginius variantus**: Pagrindinis + atsarginis įrankių modelis, kai agentas aptinka klaidas (pvz., 404 klaidas) ir automatiškai bando alternatyvų šaltinį.
- **Savi-vertinimas**: Atskirai veikiantis vertintojas-agentas, kuris įvertina atsakymus pagal pilnumą, tikslumą ir naudingumą.

Šie modeliai daro agentus tvirtesnius, skaidresnius ir patikimesnius — tai yra labai svarbūs bruožai gamybos aplinkoms.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
